# T09 Vietnamese sentiment — CPU technical demo only
Attach a **private** Kaggle input containing `ai/research/{__init__.py,data.py,pipeline.py,requirements.txt}`, `ai/configs/vietnamese_sentiment_label_map.json`, and `ai/fixtures/vietnamese_sentiment_demo.jsonl` with the same directory layout. Accelerator: None. Internet: Off. This notebook never contains research receipts, withdrawal tokens, credentials, database exports, or a real corpus. All results remain `DEMO_ONLY`; scores do not measure model quality.

In [ ]:
from pathlib import Path
import importlib.metadata, json, os, subprocess, sys
input_root = Path(os.environ.get('T09_INPUT_ROOT', '/kaggle/input')).resolve()  # override only for local package smoke
matches = sorted(input_root.rglob('ai/research/pipeline.py'))
if len(matches) != 1:
    raise RuntimeError(f'Expected exactly one ai/research/pipeline.py under {input_root}; found {len(matches)}. Attach one private T09 package.')
root = matches[0].parents[2]
required = ('ai/research/__init__.py', 'ai/research/data.py', 'ai/research/pipeline.py', 'ai/research/requirements.txt', 'ai/configs/vietnamese_sentiment_label_map.json', 'ai/fixtures/vietnamese_sentiment_demo.jsonl')
missing = [name for name in required if not (root / name).is_file()]
if missing:
    raise FileNotFoundError(f'T09 package incomplete: {missing}')
print('Python:', sys.version.split()[0])
for package in ('numpy', 'scikit-learn', 'joblib', 'matplotlib'):
    try:
        print(package + ':', importlib.metadata.version(package))
    except importlib.metadata.PackageNotFoundError as exc:
        raise RuntimeError(f'Missing {package}; install/check ai/research/requirements.txt before running') from exc
work = Path(os.environ.get('T09_OUTPUT_ROOT', '/kaggle/working/t09_demo')).resolve()
if input_root == Path('/kaggle/input') and not work.is_relative_to(Path('/kaggle/working')):
    raise RuntimeError('Kaggle output must be a new directory under /kaggle/working')
if work.exists():
    raise FileExistsError(f'Output directory exists: {work}. Set T09_OUTPUT_ROOT to a NEW path under /kaggle/working; nothing was deleted or overwritten.')
work.mkdir(parents=True)
def run(*parts):
    command = [sys.executable, '-m', 'ai.research.pipeline', *map(str, parts)]
    subprocess.run(command, cwd=root, check=True)
print('T09 DEMO_ONLY source ready; no research data loaded')

In [ ]:
snapshot = work / 'snapshot'
run('prepare', '--mode', 'demo', '--output-dir', snapshot)
selection = work / 'selection.json'
run('select', '--manifest', snapshot / 'manifest.json', '--output', selection)
evaluation = work / 'evaluation'
run('evaluate', '--manifest', snapshot / 'manifest.json', '--selection', selection, '--output-dir', evaluation)

In [ ]:
result = json.loads((evaluation / 'evaluation.json').read_text(encoding='utf-8'))
assert result['data_mode'] == 'DEMO_ONLY'
assert result['quality_claim_allowed'] is False and result['production_eligible'] is False
assert result['reload_parity'] is True
print('DEMO_ONLY local pipeline and reload parity:', result['reload_parity'])
print('Artifacts:', sorted(p.name for p in evaluation.iterdir()))